# Game Analytics Agent - Exploratory Data Analysis (EDA)

This notebook contains the exploratory data analysis and preprocessing steps for the Steam game dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
import os

# Load the datasets
game_data = pd.read_csv('data/game_data.csv')
additional_data = pd.read_csv('data/additional_data.csv')

print(f"Game Data shape: {game_data.shape}")
print(f"Additional Data shape: {additional_data.shape}")

In [ ]:
# Merging datasets
df = pd.merge(game_data, additional_data, left_on='steam_appid', right_on='appid', suffixes=('_main', '_extra'))
print(f"Merged shape: {df.shape}")

In [ ]:
# Data cleaning and preprocessing
cols_to_keep = [
    'name_main', 'steam_appid', 'is_free', 'supported_languages', 'developers', 
    'publishers', 'categories', 'genres', 'release_date', 
    'positive', 'negative', 'price', 'tags'
]
df = df[cols_to_keep].rename(columns={'name_main': 'name', 'price': 'price_usd'})

# Rating calculation
df['total_ratings'] = df['positive'] + df['negative']
df['rating'] = (df['positive'] / df['total_ratings']) * 100
df['rating'] = df['rating'].fillna(0)

# Price conversion (Roughly 1 USD = 83 INR)
df['price_inr'] = df['price_usd'] * 83

# Parse release year
def extract_year(x):
    try:
        if pd.isna(x): return np.nan
        d = ast.literal_eval(x)
        date_str = d.get('date', '')
        return int(date_str.split(',')[-1].strip())
    except:
        return np.nan

df['release_year'] = df['release_date'].apply(extract_year)

# Save cleaned data for the agent
df.to_csv('data/cleaned_game_data.csv', index=False)
df.head()

## Analytical Questions

In [ ]:
# Q1: Do free games have higher ratings on average than paid games?
avg_rating = df.groupby('is_free')['rating'].mean()
print(f"Average Rating (Free vs Paid):\n{avg_rating}")

# Q2: How many Action games support Korean?
action_korean = df[(df['genres'].str.contains('Action', na=False)) & (df['supported_languages'].str.contains('Korean', na=False))]
print(f"Action games supporting Korean: {len(action_korean)}")